In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold, train_test_split

np.random.seed(42)

In [ ]:
load_dotenv(override=True)

CONFIGS_DIR = os.getenv("CONFIGS_DIR")
DATA_DIR = os.getenv("DATA_DIR")
RESULTS_DIR = os.getenv("RESULTS_DIR")

In [ ]:
with open(os.path.join(CONFIGS_DIR, "data.yaml")) as f:
    data_config = yaml.safe_load(f)

In [ ]:
df = pd.read_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["preprocessed_data_path"])
)

In [ ]:
split_pct = data_config["preprocessing"]["split"]
target_col = data_config["columns"]["target"]

In [ ]:
df_train_valid, df_test = train_test_split(
    df, test_size=split_pct["test"], random_state=42, stratify=df[target_col]
)

df_train, df_valid = train_test_split(
    df_train_valid,
    test_size=split_pct["valid"] / (split_pct["train"] + split_pct["valid"]),
    random_state=42,
    stratify=df_train_valid[target_col],
)

print(f"Train set size: {len(df_train)}")
print(f"Validation set size: {len(df_valid)}")
print(f"Test set size: {len(df_test)}")

In [ ]:
split_index_dict = {
    "train": df_train.index,
    "valid": df_valid.index,
    "test": df_test.index,
    "train_valid": df_train_valid.index,
}

In [ ]:
with open(
    os.path.join(DATA_DIR, data_config["paths"]["split_indices_path"]), "wb"
) as f:
    pd.to_pickle(split_index_dict, f)

# Cross validation split

In [ ]:
crossval_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
crossval_positions = list(
    crossval_split.split(df_train_valid, df_train_valid[target_col])
)

In [ ]:
crossval_indices = [
    {
        "train": df_train_valid.index[train_pos],
        "valid": df_train_valid.index[valid_pos],
    }
    for train_pos, valid_pos in crossval_positions
]

In [ ]:
with open(
    os.path.join(DATA_DIR, data_config["paths"]["crossval_indices_path"]), "wb"
) as f:
    pickle.dump(crossval_indices, f)